# Track-reconstruction optimization with factored-sum loss

5-stage pipeline mirroring `tracking_opt_development_likelihood.ipynb`, with
Stage 4 (Adam refinement) using the **factored-sum** loss instead of the older
3-term `√(c·t·v)` combiner. Vertex_loss + dynamic τ_vtx are dropped — not needed
with the joint formulation.

Stages 0–3 are kept as-is (cheap pre-conditioning):

- Stage 0 — energy scan at origin (`energy_loss`)
- Stage 1 — hierarchical (position, t0) grid search (geometric `origin_time_loss`)
- Stage 2 — cone direction search (`poisson_nll` on charges)
- Stage 3 — energy scan refinement (`energy_loss`)
- **Stage 4 — Adam refinement using `factored_sum`** (the change)

This notebook is intentionally lean. Cell 1 sets knobs; cell 2 imports the
helpers; cell 3 runs N events; cell 4 prints aggregate residuals. For the full
visualization/grid-search/save machinery, see the original
`tracking_opt_development_likelihood.ipynb`.


## Setup


In [ ]:
import sys, time
sys.path.append('..')
import jax, jax.numpy as jnp
import numpy as np
import optax
from pathlib import Path

from parameter_scans_1D_gaussian import (
    DEFAULT_JSON_FILENAME, PHYSICS_CONFIG, K_PRED, K_DATA,
    NPHOT_PRED, NPHOT_DATA, SIGMA_TTS_NS,
    make_loss_fn, _MODE_PARAMS,
)
import multi_event_bias_joint as meb
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.optimization.grid_search import (
    load_optimization_config, get_detector_bounds,
    hierarchical_position_grid_search,
)
from lucid.optimization.utils.functions import (
    cartesian_to_spherical, spherical_to_cartesian,
)

# Knobs
MODE = 'factored_sum'
HIT_THRESHOLD = 2.0
LAMBDA_BG = 1e-3
WAVELENGTH_MODE = False
N_EVENTS = 10
SEED_BASE = 44
START_ENTRY = 0
MAX_ITERATIONS = 400
TOLERANCE = 1e-6
print(f'mode={MODE}, thr={HIT_THRESHOLD}, lambda_bg={LAMBDA_BG}, '
      f'wavelength_mode={WAVELENGTH_MODE}, N_EVENTS={N_EVENTS}')


In [ ]:
# Build sims
detector = generate_detector(DEFAULT_JSON_FILENAME)
detector_points = jnp.array(detector.all_points)
num_detectors = len(detector_points)
detector_bounds = get_detector_bounds(detector)

data_sim = setup_event_simulator(
    DEFAULT_JSON_FILENAME, NPHOT_DATA,
    temperature=None, K=K_DATA,
    is_data=True, is_calibration=False, apply_smearing=True,
    wavelength_mode=WAVELENGTH_MODE,
    physics_config=PHYSICS_CONFIG, default_detector_params=True)
pred_sim = setup_event_simulator(
    DEFAULT_JSON_FILENAME, NPHOT_PRED, 0.10,
    max_candidates_per_ray=4, K=K_PRED,
    is_data=False, hit_mode='per_photon',
    wavelength_mode=WAVELENGTH_MODE,
    physics_config=PHYSICS_CONFIG, default_detector_params=True)

loss_fn = make_loss_fn(pred_sim, num_detectors)

# Stage 4 loss: factored_sum via flag setup
cw, tw, gm, jt, fac = _MODE_PARAMS[MODE]
cw_j, tw_j, gm_j, jt_j, fac_j = (jnp.float32(v) for v in (cw, tw, gm, jt, fac))
thr_j = jnp.float32(HIT_THRESHOLD)
bg_j = jnp.float32(LAMBDA_BG)

@jax.jit
def loss_and_grad_s4(params, hit_times, hit_counts, key):
    def f(p):
        return loss_fn(p, hit_times, hit_counts, key,
                       cw_j, tw_j, thr_j, bg_j, gm_j, jt_j, fac_j)
    return jax.value_and_grad(f)(params)

print('Sims + Stage-4 loss ready.')


## Stages 0–3 (pre-conditioning, simple losses)

These follow the reference notebook. Energy scan at origin, hierarchical
position+t0 grid, cone direction search, energy refinement. We only need
modest accuracy here — Stage 4 polishes.


In [ ]:
# Light wrappers for the reference-notebook stages. We use Poisson + energy_loss
# heuristics for stages 0–3 (consistent with tracking_opt_development_likelihood).

def energy_loss(sim_counts, true_counts, eps=1e-8):
    return jnp.abs(jnp.log(jnp.sum(sim_counts) / (jnp.sum(true_counts) + eps)))

def poisson_nll_charge(true, pred, eps=1e-8):
    nll = pred - true * jnp.log(pred + eps) + jax.scipy.special.gammaln(true + 1.0)
    return jnp.sum(nll) / (jnp.sum(true) + eps)

def stage0(observed_counts, true_energy):
    theta_i = jnp.arccos(1/jnp.sqrt(3)); phi_i = jnp.pi/4.
    pos_i = jnp.array([0., 0., 0.])
    energy_guess = 1000 + np.random.uniform(-50, 50)
    energies = jnp.linspace(energy_guess - 700, energy_guess + 700, 10)
    scan_key = jax.random.PRNGKey(42)
    best_loss, best_e = float('inf'), energy_guess
    for e in energies:
        track = ParticleParams(energy=e, position=pos_i, theta=theta_i,
                                phi=phi_i, t0=jnp.array(0.0))
        _, _, _, q = pred_sim(track, scan_key)
        L = float(energy_loss(q, observed_counts))
        if L < best_loss: best_loss, best_e = L, e
    return float(best_e)

def stage2(opt_pos, opt_t0, energy_guess, observed_counts, true_direction,
            levels=3, divs=8, max_angle_deg=180., reduction=0.5):
    best_dir = np.array([0., 0., 1.]); best_th = 0.; best_ph = 0.; best_loss = float('inf')
    cone_key = jax.random.PRNGKey(42)
    cur_max = np.radians(max_angle_deg)
    for level in range(levels):
        n_t, n_p = divs, divs * 2
        if level == 0:
            for i in range(n_t):
                tv = np.pi * (i / max(n_t - 1, 1))
                for j in range(n_p):
                    pv = 2 * np.pi * (j / n_p)
                    track = ParticleParams(energy=energy_guess, position=opt_pos,
                                            theta=tv, phi=pv, t0=opt_t0)
                    _, _, _, q = pred_sim(track, cone_key)
                    L = float(poisson_nll_charge(observed_counts, q))
                    if L < best_loss:
                        best_loss = L
                        best_dir = np.array(spherical_to_cartesian(tv, pv))
                        best_th, best_ph = tv, pv
        cur_max *= reduction
    cos_a = np.clip(np.dot(best_dir, true_direction), -1., 1.)
    return float(best_th), float(best_ph), float(np.degrees(np.arccos(cos_a)))

def stage3(opt_pos, th, ph, t0_v, energy_guess, observed_counts, n_steps=10, delta=400):
    energies = jnp.linspace(energy_guess - delta, energy_guess + delta, n_steps)
    scan_key = jax.random.PRNGKey(42)
    best_loss, best_e = float('inf'), energy_guess
    for e in energies:
        track = ParticleParams(energy=e, position=opt_pos, theta=th, phi=ph, t0=t0_v)
        _, _, _, q = pred_sim(track, scan_key)
        L = float(energy_loss(q, observed_counts))
        if L < best_loss: best_loss, best_e = L, e
    return float(best_e)

print('Stage 0/2/3 helpers ready.')


## Stage 4 — Adam refinement with factored-sum loss


In [ ]:
def stage4(initial_params, hit_times, hit_counts, true_pos, true_dir, TRUE_T0,
            true_energy, lr=0.2, b1=0.9, b2=0.999, eps_a=1e-8):
    opt = optax.adam(learning_rate=lr, b1=b1, b2=b2, eps=eps_a)
    opt_state = opt.init(initial_params)
    cur = jnp.array(initial_params)
    POS_LR, DIR_LR, T0_LR, ENE_LR = 0.4, 0.5, 0.05, 1.0
    opt_key = jax.random.PRNGKey(12345)
    DETR = detector_bounds['r']; DETH = detector_bounds['H']
    history = {'losses': [], 'pos_err': [], 'dir_err': [], 't0_err': [], 'E_err': []}
    for it in range(MAX_ITERATIONS):
        opt_key, _ = jax.random.split(opt_key)
        L, g = loss_and_grad_s4(cur, hit_times, hit_counts, opt_key)
        if jnp.any(jnp.isnan(g)):
            g = jnp.nan_to_num(g, nan=0.0)
        if jnp.linalg.norm(g) < TOLERANCE:
            break
        if it < 25:
            scales = jnp.array([0., 0., 0., 0., DIR_LR, DIR_LR, 0.])
        else:
            scales = jnp.array([POS_LR, POS_LR, POS_LR, T0_LR, DIR_LR, DIR_LR, ENE_LR])
        upd, opt_state = opt.update(g, opt_state, cur)
        cur = optax.apply_updates(cur, upd * scales)
        cur = jnp.array([
            jnp.clip(cur[0], -DETR*0.95, DETR*0.95),
            jnp.clip(cur[1], -DETR*0.95, DETR*0.95),
            jnp.clip(cur[2], -DETH/2*0.95, DETH/2*0.95),
            jnp.clip(cur[3], -20., 20.),
            cur[4], cur[5],
            jnp.clip(cur[6], 300., 2000.)])
        cur_dir = spherical_to_cartesian(cur[4], cur[5])
        cos_a = np.clip(np.dot(np.array(cur_dir), np.array(true_dir)), -1., 1.)
        history['losses'].append(float(L))
        history['pos_err'].append(float(jnp.linalg.norm(cur[:3] - true_pos)))
        history['dir_err'].append(float(np.degrees(np.arccos(cos_a))))
        history['t0_err'].append(float(abs(cur[3] - TRUE_T0)))
        history['E_err'].append(float(abs(cur[6] - true_energy)))
    return cur, history
print('Stage 4 ready.')


## Run N events


In [ ]:
all_results = []
for ev in range(N_EVENTS):
    entry_idx = START_ENTRY + ev
    seed = SEED_BASE + ev
    print(f'\n=== event {ev + 1}/{N_EVENTS}  (entry={entry_idx}, seed={seed}) ===')
    (true_track, true_data, x, y, z, theta, phi, energy) = meb.build_event_for_entry(
        detector, data_sim, num_detectors, entry_idx=entry_idx, seed=seed)
    TRUE_T0 = 0.0
    true_pos = np.array([x, y, z])
    true_dir = np.asarray(spherical_to_cartesian(theta, phi))
    n_hit = int(np.sum(np.asarray(true_data[0]) > 0))
    print(f'  true: pos=({x:+.2f},{y:+.2f},{z:+.2f}), E={energy:.0f}, n_hit={n_hit}')

    hit_counts, hit_times = true_data
    obs_counts = np.asarray(hit_counts)
    obs_times = hit_times

    # Stage 0
    e0 = stage0(obs_counts, energy)
    # Stage 1 (geometric grid)
    hit_mask = obs_counts > 0
    hit_positions = detector_points[hit_mask]
    obs_times_hit = obs_times[hit_mask]
    obs_counts_hit = hit_counts[hit_mask]
    s1 = hierarchical_position_grid_search(
        hit_positions, obs_times_hit, obs_counts_hit,
        true_pos, TRUE_T0, 0.0, detector_bounds,
        n_div=5, t0_n_div=5, levels=4, fraction=0.95,
        t0_min=-15., t0_max=15., min_L=0.05, verbosity=0)
    # Stage 2 (direction cone)
    th2, ph2, dir_err = stage2(s1['best_position'], s1['best_t0'], e0,
                                  obs_counts, true_dir)
    # Stage 3 (energy refine)
    e3 = stage3(s1['best_position'], th2, ph2, s1['best_t0'], e0, obs_counts)

    # Stage 4 (Adam, factored_sum)
    init = jnp.array([s1['best_position'][0], s1['best_position'][1],
                       s1['best_position'][2], s1['best_t0'], th2, ph2, e3])
    final, hist = stage4(init, hit_times, hit_counts,
                            true_pos, true_dir, TRUE_T0, energy)
    print(f'  final: pos_err={hist["pos_err"][-1]:.3f}m  '
          f'dir_err={hist["dir_err"][-1]:.2f}°  '
          f't0_err={hist["t0_err"][-1]:.3f}  '
          f'E_err={hist["E_err"][-1]:.1f}MeV  ({len(hist["losses"])} iters)')
    all_results.append({'event_idx': ev, 'entry_idx': entry_idx, 'seed': seed,
                          'final': np.asarray(final), 'history': hist,
                          'true_pos': true_pos, 'true_dir': true_dir,
                          'true_energy': float(energy)})


## Aggregate residuals


In [ ]:
pos_errs = np.array([r['history']['pos_err'][-1] for r in all_results])
dir_errs = np.array([r['history']['dir_err'][-1] for r in all_results])
t0_errs  = np.array([r['history']['t0_err'][-1] for r in all_results])
E_errs   = np.array([r['history']['E_err'][-1] for r in all_results])
E_signed = np.array([r['final'][6] - r['true_energy'] for r in all_results])

def stats(a):
    return f'mean={np.mean(a):+.3f}, median={np.median(a):+.3f}, std={np.std(a, ddof=1):.3f}'
print(f'  pos error (m):   {stats(pos_errs)}')
print(f'  dir error (°):   {stats(dir_errs)}')
print(f'  t0 error (ns):   {stats(t0_errs)}')
print(f'  E error (MeV):   {stats(E_errs)}')
print(f'  E signed (MeV):  {stats(E_signed)}    # negative = under-estimated')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, arr, title in zip(axes.ravel(),
                                [pos_errs, dir_errs, t0_errs, E_signed],
                                ['pos error (m)', 'dir error (deg)',
                                 't0 error (ns)', 'E signed (MeV)']):
    ax.hist(arr, bins=15, color='C0', alpha=0.7, edgecolor='black')
    ax.axvline(0, color='red', ls='--')
    ax.axvline(arr.mean(), color='darkorange', ls='-',
                label=f'mean={arr.mean():+.3f}')
    ax.set_title(title); ax.set_ylabel('events')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Notes

- Stage 4 uses the **factored-sum** loss (C_event + T_event), with all knobs
  (`MODE`, `HIT_THRESHOLD`, `LAMBDA_BG`, `WAVELENGTH_MODE`) exposed at the top.
- Stages 0–3 are kept simple/cheap as in the reference notebook.
- For comparison, change `MODE` to `'joint'` or `'gmean_joint'` to see how
  the converged residuals change.
- Per-event traces are in `all_results[i]['history']` (loss, pos_err, etc.
  per Adam iteration).
